# Week 2 Exercise: Technical Mentor AI

**Use-Case:** This is a **Technical Mentor Chatbot** for junior developers and data scientists. It acts as an expert tutor that explains complex coding and machine learning concepts.

**Features:**
- **Model Switching:** Switch between `gpt-4o-mini` and local `llama3.2`.
- **Persona:** System prompt sets an expert technical tutor persona.
- **Streaming:** Returns responses progressively.
- **Tools:** A glossary tool fetches exact definitions for specific terms.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json

load_dotenv(override=True)
openai_client = OpenAI()
ollama_client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
glossary = {"distillation": "Training a smaller model to reproduce a larger model.", "rag": "Retrieval-Augmented Generation"}

def lookup_definition(term):
    return glossary.get(term.lower(), "Definition not found.")

tools = [{"type": "function", "function": {"name": "lookup_definition", "description": "Look up a technical term", "parameters": {"type": "object", "properties": {"term": {"type": "string"}}, "required": ["term"]}}}]

In [ ]:
system_message = "You are a concise technical tutor. Use your tool to look up technical definitions."

def chat(message, history, model_choice):
    hist = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + hist + [{"role": "user", "content": message}]
    client = openai_client if model_choice == "gpt-4o-mini" else ollama_client
    model = "gpt-4o-mini" if model_choice == "gpt-4o-mini" else "llama3.2"
    use_tools = tools if model_choice == "gpt-4o-mini" else None

    if use_tools:
        response = client.chat.completions.create(model=model, messages=messages, tools=use_tools, stream=False)
        if response.choices[0].finish_reason == "tool_calls":
            msg = response.choices[0].message
            messages.append(msg)
            for tool_call in msg.tool_calls:
                messages.append({"role": "tool", "content": lookup_definition(json.loads(tool_call.function.arguments).get('term')), "tool_call_id": tool_call.id})
            stream = client.chat.completions.create(model=model, messages=messages, stream=True)
            resp = ""
            for chunk in stream:
                resp += chunk.choices[0].delta.content or ''
                yield resp
            return

    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    resp = ""
    for chunk in stream:
        resp += chunk.choices[0].delta.content or ''
        yield resp

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("#Technical Mentor AI")
    model_dropdown = gr.Dropdown(choices=["gpt-4o-mini", "llama3.2"], value="gpt-4o-mini")
    gr.ChatInterface(fn=chat, type="messages", additional_inputs=[model_dropdown])

if __name__ == "__main__":
    demo.launch()